## 1. Initialize Project Environment
Import libraries for sequence fetching, distance calculation, and matrix formatting.

In [1]:
from __future__ import annotations

import itertools
import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
print("numpy", np.__version__)
try:
    import Bio

    print("biopython", Bio.__version__)
except Exception as exc:
    logging.error("Biopython import failed: %s", exc)

pandas 2.2.3
numpy 2.1.3
biopython 1.85


## 2. Define Configuration Parameters
Centralize sequence accessions, export paths, and distance calculation options.

In [2]:
@dataclass
class DistanceConfig:
    handle: str
    email: str
    export_dir: Path = Path("artifacts")
    truncate_to_min: bool = True
    accessions: List[str] = None

    def __post_init__(self):
        if self.accessions is None:
            # TP53 sequences from 10+ different organisms
            self.accessions = [
                "NM_000546.6",  # Homo sapiens (human)
                "NM_011640.3",  # Mus musculus (mouse)
                "NM_131327.2",  # Danio rerio (zebrafish)
                "XM_006719566.3",  # Canis lupus familiaris (dog)
                "NM_001317019.1",  # Sus scrofa (pig)
                "NM_001085860.1",  # Bos taurus (cattle)
                "XM_005194938.2",  # Callicebus moloch (dusky titi)
                "NM_001006919.1",  # Anas platyrhynchos (duck)
                "NM_001089263.1",  # Gallus gallus (chicken)
                "XM_031279688.1",  # Panthera leo (lion)
            ]

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = DistanceConfig(
    handle="AndreiCod",
    email="student@example.com",
)
CONFIG.describe()

{'handle': 'AndreiCod',
 'email': 'student@example.com',
 'export_dir': 'artifacts',
 'truncate_to_min': True,
 'accessions': ['NM_000546.6',
  'NM_011640.3',
  'NM_131327.2',
  'XM_006719566.3',
  'NM_001317019.1',
  'NM_001085860.1',
  'XM_005194938.2',
  'NM_001006919.1',
  'NM_001089263.1',
  'XM_031279688.1']}

In [3]:
def fetch_sequences(cfg: DistanceConfig) -> List[SeqRecord]:
    """Fetch sequences from NCBI Entrez."""
    Entrez.email = cfg.email
    records = []

    for accession in cfg.accessions:
        try:
            logging.info("Fetching %s...", accession)
            handle = Entrez.efetch(
                db="nucleotide", id=accession, rettype="fasta", retmode="text"
            )
            record = SeqIO.read(handle, "fasta")
            handle.close()
            records.append(record)
        except Exception as e:
            logging.warning("Failed to fetch %s: %s", accession, e)

    logging.info("Successfully fetched %d sequences", len(records))
    return records


# Fetch all sequences
sequences = fetch_sequences(CONFIG)
print(f"\nFetched {len(sequences)} sequences:")
for rec in sequences:
    print(f"  {rec.id}: {len(rec.seq)} bp")

[INFO] Fetching NM_000546.6...
[INFO] Fetching NM_011640.3...
[INFO] Fetching NM_131327.2...
[INFO] Fetching XM_006719566.3...
[INFO] Fetching NM_001317019.1...
[INFO] Fetching NM_001085860.1...
[INFO] Fetching XM_005194938.2...
[INFO] Fetching NM_001006919.1...
[INFO] Fetching NM_001089263.1...
[INFO] Fetching XM_031279688.1...
[INFO] Successfully fetched 10 sequences



Fetched 10 sequences:
  NM_000546.6: 2512 bp
  NM_011640.3: 1781 bp
  NM_131327.2: 2233 bp
  XM_006719566.3: 3347 bp
  NM_001317019.1: 4747 bp
  NM_001085860.1: 1719 bp
  XM_005194938.2: 5035 bp
  NM_001006919.1: 1515 bp
  NM_001089263.1: 2415 bp
  XM_031279688.1: 808 bp


In [4]:
# Save sequences for later use
DATA_DIR = Path(f"../../../data/work/{CONFIG.handle}/lab04")
DATA_DIR.mkdir(parents=True, exist_ok=True)
fasta_path = DATA_DIR / "tp53_multi_sequences.fasta"
SeqIO.write(sequences, fasta_path, "fasta")
logging.info("Saved %d sequences to %s", len(sequences), fasta_path)

[INFO] Saved 10 sequences to ../../../data/work/AndreiCod/lab04/tp53_multi_sequences.fasta


## 3. Implement Core Functionality
Compute Hamming and p-distance scores, format as upper-triangular matrix.

In [5]:
def truncate_pair(a: str, b: str, enabled: bool = True) -> Tuple[str, str, int]:
    """Truncate sequences to equal length for comparison."""
    if not enabled:
        return a, b, len(a)
    L = min(len(a), len(b))
    return a[:L], b[:L], L


def hamming_distance(a: str, b: str) -> int:
    """Compute Hamming distance between two equal-length strings."""
    if len(a) != len(b):
        raise ValueError("Hamming distance requires equal-length strings")
    return sum(ch1 != ch2 for ch1, ch2 in zip(a, b))


def compute_pairwise_distances(
    records: List[SeqRecord], truncate: bool = True
) -> pd.DataFrame:
    """Compute all pairwise distances between sequences."""
    rows = []
    for rec_i, rec_j in itertools.combinations(records, 2):
        seq_i, seq_j, used_len = truncate_pair(
            str(rec_i.seq), str(rec_j.seq), enabled=truncate
        )
        d_hamming = hamming_distance(seq_i, seq_j)
        d_p = d_hamming / used_len if used_len else np.nan
        rows.append(
            {
                "seq_a": rec_i.id,
                "seq_b": rec_j.id,
                "used_len": used_len,
                "hamming": d_hamming,
                "p_distance": d_p,
            }
        )
    return pd.DataFrame(rows)


pairwise_df = compute_pairwise_distances(sequences, truncate=CONFIG.truncate_to_min)
pairwise_df

,seq_a,seq_b,used_len,hamming,p_distance
0,NM_000546.6,NM_011640.3,1781,1286,0.722066
1,NM_000546.6,NM_131327.2,2233,1681,0.752799
2,NM_000546.6,XM_006719566.3,2512,1908,0.759554
3,NM_000546.6,NM_001317019.1,2512,1898,0.755573
4,NM_000546.6,NM_001085860.1,1719,1293,0.752182
5,NM_000546.6,XM_005194938.2,2512,1891,0.752787
6,NM_000546.6,NM_001006919.1,1515,1148,0.757756
7,NM_000546.6,NM_001089263.1,2415,1775,0.734990
8,NM_000546.6,XM_031279688.1,808,585,0.724010
9,NM_011640.3,NM_131327.2,1781,1293,0.725997


In [6]:
def create_distance_matrix(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """Create symmetric distance matrix from pairwise dataframe."""
    ids = sorted(set(df["seq_a"]) | set(df["seq_b"]))
    matrix = pd.DataFrame(0.0, index=ids, columns=ids)

    for _, row in df.iterrows():
        matrix.loc[row["seq_a"], row["seq_b"]] = row[value_col]
        matrix.loc[row["seq_b"], row["seq_a"]] = row[value_col]

    return matrix


p_distance_matrix = create_distance_matrix(pairwise_df, "p_distance")
p_distance_matrix

,NM_000546.6,NM_001006919.1,NM_001085860.1,NM_001089263.1,NM_001317019.1,NM_011640.3,NM_131327.2,XM_005194938.2,XM_006719566.3,XM_031279688.1
NM_000546.6,0.000000,0.757756,0.752182,0.734990,0.755573,0.722066,0.752799,0.752787,0.759554,0.724010
NM_001006919.1,0.757756,0.000000,0.765677,0.722772,0.763696,0.766997,0.751155,0.763696,0.736634,0.742574
NM_001085860.1,0.752182,0.765677,0.000000,0.766725,0.751018,0.752763,0.755090,0.742874,0.757999,0.775990
NM_001089263.1,0.734990,0.722772,0.766725,0.000000,0.744099,0.765300,0.754590,0.761905,0.728778,0.764851
NM_001317019.1,0.755573,0.763696,0.751018,0.744099,0.000000,0.733857,0.746082,0.733727,0.736480,0.751238
NM_011640.3,0.722066,0.766997,0.752763,0.765300,0.733857,0.000000,0.725997,0.742841,0.747894,0.742574
NM_131327.2,0.752799,0.751155,0.755090,0.754590,0.746082,0.725997,0.000000,0.753247,0.740260,0.745050
XM_005194938.2,0.752787,0.763696,0.742874,0.761905,0.733727,0.742841,0.753247,0.000000,0.747535,0.759901
XM_006719566.3,0.759554,0.736634,0.757999,0.728778,0.736480,0.747894,0.740260,0.747535,0.000000,0.757426
XM_031279688.1,0.724010,0.742574,0.775990,0.764851,0.751238,0.742574,0.745050,0.759901,0.757426,0.000000


## 4. Validate with Unit Tests
Quick assertions to verify distance calculations are correct.

In [7]:
def test_hamming_distance():
    assert hamming_distance("AAAA", "AAAT") == 1
    assert hamming_distance("AC", "GT") == 2
    assert hamming_distance("ACGT", "ACGT") == 0


def test_truncate_pair():
    a, b, used = truncate_pair("AAAA", "AA", True)
    assert used == 2 and a == "AA" and b == "AA"


def test_pairwise_distances():
    dummy = [
        SeqRecord(seq=Seq("AAAA"), id="A"),
        SeqRecord(seq=Seq("AAAT"), id="B"),
        SeqRecord(seq=Seq("AATT"), id="C"),
    ]
    df = compute_pairwise_distances(dummy, truncate=True)
    assert len(df) == 3
    assert set(df.columns) == {"seq_a", "seq_b", "used_len", "hamming", "p_distance"}


test_hamming_distance()
test_truncate_pair()
test_pairwise_distances()
logging.info("All inline tests passed.")

[INFO] All inline tests passed.


## 5. Export Results
Save distance matrix and pairwise data to artifacts folder.

In [8]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save pairwise distances
pairwise_path = EXPORT_DIR / "task1_pairwise_distances.csv"
pairwise_df.to_csv(pairwise_path, index=False)
print(f"[OK] Pairwise distances saved to: {pairwise_path.resolve()}")

# Save distance matrix
matrix_path = EXPORT_DIR / "task1_distance_matrix.csv"
p_distance_matrix.to_csv(matrix_path)
print(f"[OK] Distance matrix saved to: {matrix_path.resolve()}")

print(f"\nArtifacts saved to {EXPORT_DIR.resolve()}")

[OK] Pairwise distances saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task1_pairwise_distances.csv
[OK] Distance matrix saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task1_distance_matrix.csv

Artifacts saved to /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts
